In [1]:
import numpy as np
from sklearn.base import ClassifierMixin, BaseEstimator
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import check_X_y, check_array
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class BaggingClassifier(BaseEstimator, ClassifierMixin):
    """
    Bagging classifier built from scratch.

    Parameters
    ----------
    base_estimator : estimator object, default=None
        The base model to be bagged. If None, uses DecisionTreeClassifier.
    n_estimators : int, default=10
        Number of models in the ensemble.
    max_samples : float or int, default=1.0
        The number of samples to draw from X to train each base estimator.
        - If float, it is a fraction of the total number of samples.
        - If int, it is the absolute number.
    bootstrap : bool, default=True
        Whether to sample with replacement (True) or without (False).
    random_state : int, default=None
        Seed for reproducible bootstrap sampling.
    """
    def __init__(self,
                 base_estimator=None,
                 n_estimators=10,
                 max_samples=1.0,
                 bootstrap=True,
                 random_state=None):
        self.base_estimator = base_estimator or DecisionTreeClassifier()
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.bootstrap = bootstrap
        self.random_state = random_state
        self.estimators_ = []
        self.sample_indices_ = []

    def fit(self, X, y):
        """Fit the bagging ensemble."""
        X, y = check_X_y(X, y)
        rng = np.random.RandomState(self.random_state)
        n_samples = X.shape[0]

        # Determine the size of each bootstrap sample
        if isinstance(self.max_samples, float):
            sample_size = int(self.max_samples * n_samples)
        else:
            sample_size = int(self.max_samples)
        sample_size = min(sample_size, n_samples)

        self.classes_ = np.unique(y)
        self.estimators_ = []
        self.sample_indices_ = []

        for _ in range(self.n_estimators):
            # Bootstrap sample (with or without replacement)
            if self.bootstrap:
                indices = rng.choice(n_samples, size=sample_size, replace=True)
            else:
                indices = rng.choice(n_samples, size=sample_size, replace=False)
            X_sample = X[indices]
            y_sample = y[indices]

            # Train a fresh copy of the base estimator
            estimator = self.base_estimator.__class__(**self.base_estimator.get_params())
            estimator.fit(X_sample, y_sample)

            self.estimators_.append(estimator)
            self.sample_indices_.append(indices)

        return self

    def predict(self, X):
        """Predict class for X using majority vote."""
        X = check_array(X)
        # Collect predictions from all estimators: shape (n_estimators, n_samples)
        all_preds = np.array([est.predict(X) for est in self.estimators_])

        # Majority vote (mode) along the estimator axis
        preds = []
        for col in range(all_preds.shape[1]):
            vals, counts = np.unique(all_preds[:, col], return_counts=True)
            preds.append(vals[np.argmax(counts)])
        return np.array(preds)

    def predict_proba(self, X):
        """Predict class probabilities by averaging probabilities from each model."""
        X = check_array(X)
        # Average probabilities (base estimator must support predict_proba)
        probas = np.mean([est.predict_proba(X) for est in self.estimators_], axis=0)
        return probas




if __name__ == "__main__":
    # Generate synthetic classification data
    X, y = make_classification(n_samples=300, n_features=10,
                               n_informative=8, n_redundant=2,
                               random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # Create and train the bagging ensemble
    bag = BaggingClassifier(n_estimators=20, max_samples=0.8,
                            bootstrap=True, random_state=42)
    bag.fit(X_train, y_train)

    # Evaluate
    y_pred = bag.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Bagging Accuracy: {acc:.3f}")   # typically ~0.883

    # Probabilities example
    proba = bag.predict_proba(X_test[:5])
    print("Predicted probabilities (first 5 samples):\n", proba)

Bagging Accuracy: 0.822
Predicted probabilities (first 5 samples):
 [[0.45 0.55]
 [0.85 0.15]
 [0.5  0.5 ]
 [0.75 0.25]
 [0.85 0.15]]
